# NovaPay Simplified Checkout A/B Test — Guided Analysis

This notebook walks through the standard A/B testing framework using a fictional payments experiment. All data are synthetic.

**Decision:** Should NovaPay roll out a simplified checkout experience?

**Primary KPI:** checkout completion rate.  
**Secondary KPI:** checkout time.  
**Guardrails:** payment decline rate, support contact rate, fraud rate.

## Standard framework

1. Frame the business decision.
2. Define hypothesis and decision rules.
3. Define population and randomization.
4. Define primary, secondary, and guardrail KPIs.
5. Validate data quality.
6. Check experiment health and group balance.
7. Analyze the primary KPI.
8. Analyze guardrails and continuous metrics.
9. Explore pre-specified segments.
10. Translate the result into business impact and recommendation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
module_root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(module_root / 'src'))

from generate_synthetic_data import generate_raw_data
from analyze_experiment import (
    quality_report, clean_data, sample_ratio_check, kpi_summary,
    two_proportion_test, continuous_metric_analysis, segment_completion
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
ALPHA = 0.05
MIN_BUSINESS_LIFT_PP = 2.0
EXPECTED_TREATMENT_SHARE = 0.50

## 1. Frame the business question

NovaPay believes its existing checkout flow creates friction. Product proposes a simplified version.

> **Does the simplified checkout increase completion enough to justify rollout without harming payment quality, support demand, or fraud risk?**

Start with the business decision, not the statistical test.

## 2. Hypothesis and decision rules

**H0:** Checkout completion is the same in Control and Treatment.  
**H1:** Checkout completion differs between Control and Treatment.

Training thresholds:
- alpha = 0.05
- minimum business-relevant lift = +2.0 percentage points
- no material adverse guardrail movement

Statistical significance and business significance are evaluated separately.

## 3. Generate the synthetic raw data

The generator deliberately adds duplicates, missing treatment assignments, and inconsistent country casing so the analysis includes realistic data-quality work.

In [ ]:
raw = generate_raw_data()
print(f'Raw rows: {len(raw):,}')
raw.head()

## 4. Validate data quality

Before looking at treatment performance, quantify data issues. Check schema, experimental-unit uniqueness, treatment assignment, category consistency, binary fields, missingness, date coverage, and implausible values.

In [ ]:
pd.Series(quality_report(raw), name='count').to_frame()

In [ ]:
display(raw.isna().sum().to_frame('missing_rows'))
print('Date range:', raw['experiment_date'].min(), 'to', raw['experiment_date'].max())

### Cleaning rules for this case

1. Standardize country labels to uppercase.
2. Exclude invalid or missing experiment assignments.
3. Keep one record per customer.
4. Parse experiment dates.

In production, document all exclusions and quantify their impact.

In [ ]:
df = clean_data(raw)
print(f'Raw rows: {len(raw):,}')
print(f'Final analytical population: {len(df):,}')
print(f'Rows removed: {len(raw) - len(df):,}')

## 5. Experiment health: Sample Ratio Mismatch (SRM)

The intended assignment is 50/50. A large unexplained departure can indicate broken randomization, tracking problems, eligibility errors, or extraction issues.

The chi-square check below is used as an **experiment-health diagnostic**, not as the primary treatment-effect test.

In [ ]:
srm = sample_ratio_check(df, expected_treatment_share=EXPECTED_TREATMENT_SHARE)
display(pd.Series(srm).to_frame('value'))
print('PASS: no SRM evidence' if srm['p_value'] >= ALPHA else 'WARNING: investigate possible SRM')

## 6. Balance checks

Randomization makes groups comparable in expectation, but we still inspect important pre-treatment characteristics as a sanity check. Look for meaningful imbalance rather than hunting for significance across many baseline variables.

In [ ]:
def balance_table(data, column):
    return (pd.crosstab(data[column], data['experiment_group'], normalize='columns') * 100).round(2)

print('Country mix (%)'); display(balance_table(df, 'country'))
print('Device mix (%)'); display(balance_table(df, 'device_type'))
print('Tenure mix (%)'); display(balance_table(df, 'customer_tenure'))

## 7. KPI summary

Review the primary KPI, secondary metric, and guardrails together before formal inference. This exposes trade-offs early.

In [ ]:
summary = kpi_summary(df)
summary

In [ ]:
rate_cols = ['checkout_completion_rate','payment_decline_rate','support_contact_rate','fraud_rate']
ax = (summary[rate_cols].T * 100).plot(kind='bar', figsize=(9,5))
ax.set_ylabel('Rate (%)')
ax.set_title('Control vs Treatment — Rate KPIs')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

## 8. Primary KPI: checkout completion

Checkout completion is binary, so we compare two proportions. Report the absolute lift, relative lift, p-value, and 95% confidence interval. Then compare the effect with the pre-defined business threshold.

In [ ]:
primary = two_proportion_test(df, 'checkout_completed')
display(pd.Series(primary.__dict__).to_frame('value'))

lift_pp = primary.absolute_lift * 100
print(f'Control: {primary.control_rate:.2%}')
print(f'Treatment: {primary.treatment_rate:.2%}')
print(f'Absolute lift: {lift_pp:.2f} pp')
print(f'Relative lift: {primary.relative_lift:.2%}')
print(f'95% CI: [{primary.ci_low*100:.2f}, {primary.ci_high*100:.2f}] pp')
print(f'p-value: {primary.p_value:.6g}')
print('Statistically significant:', primary.p_value < ALPHA)
print('Business meaningful:', lift_pp >= MIN_BUSINESS_LIFT_PP)

### Client-facing interpretation

Do not stop at *p < 0.05*. Explain the size of the effect, uncertainty around it, and whether it clears the business threshold.

## 9. Guardrails

For payment declines, support contacts, and fraud, lower is better. Rare outcomes such as fraud need especially cautious interpretation because small event counts can create noisy relative movements.

In [ ]:
guardrails = {}
for metric in ['payment_declined','support_contact','fraud_flag']:
    r = two_proportion_test(df, metric)
    guardrails[metric] = {
        'control_rate': r.control_rate,
        'treatment_rate': r.treatment_rate,
        'absolute_change_pp': r.absolute_lift*100,
        'p_value': r.p_value,
        'ci_low_pp': r.ci_low*100,
        'ci_high_pp': r.ci_high*100
    }
pd.DataFrame(guardrails).T

## 10. Continuous metric: checkout time

Checkout time is intentionally right-skewed. Inspect mean, median, skewness, and distribution. Welch's t-test estimates a mean difference without assuming equal variances; Mann–Whitney is included as a sensitivity check.

In [ ]:
time_result = continuous_metric_analysis(df, 'checkout_time_seconds')
pd.Series(time_result).to_frame('value')

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for group in ['Control','Treatment']:
    x = df.loc[df['experiment_group']==group, 'checkout_time_seconds']
    ax.hist(x, bins=60, alpha=0.45, density=True, label=group)
ax.set_xlim(0, df['checkout_time_seconds'].quantile(.99))
ax.set_xlabel('Checkout time (seconds)')
ax.set_ylabel('Density')
ax.set_title('Checkout Time Distribution — display trimmed at 99th percentile')
ax.legend(); plt.tight_layout(); plt.show()

## 11. Segment analysis

The synthetic data contain a stronger treatment response on Mobile. Segment findings should be labeled as pre-specified or post-hoc. Post-hoc patterns generate hypotheses; they do not automatically become causal conclusions.

In [ ]:
device = segment_completion(df, 'device_type')
display(device)
ax = (device[['control_rate','treatment_rate']] * 100).plot(kind='bar', figsize=(7,4))
ax.set_ylabel('Checkout completion (%)')
ax.set_title('Checkout Completion by Device')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
print('Country'); display(segment_completion(df, 'country'))
print('Customer tenure'); display(segment_completion(df, 'customer_tenure'))

## 12. Translate into business impact

Assume 2 million eligible checkout attempts annually. This simple scaling illustrates how an experiment effect becomes a business conversation. A real ROI case would also request payment volume, margin, fraud cost, servicing cost, and implementation cost.

In [ ]:
ANNUAL_ELIGIBLE_CHECKOUTS = 2_000_000
incremental = ANNUAL_ELIGIBLE_CHECKOUTS * primary.absolute_lift
print(f'Illustrative annual eligible checkouts: {ANNUAL_ELIGIBLE_CHECKOUTS:,}')
print(f'Estimated incremental completed checkouts: {incremental:,.0f}')

## 13. Decision framework

A recommendation should integrate:

1. **Experiment validity** — can we trust the design and data?
2. **Statistical evidence** — is the difference distinguishable from random variation?
3. **Business magnitude** — is the effect large enough to matter?
4. **Guardrails** — did the treatment create unacceptable harm?

In [ ]:
srm_ok = srm['p_value'] >= ALPHA
primary_ok = primary.p_value < ALPHA and lift_pp >= MIN_BUSINESS_LIFT_PP
adverse_guardrails = [
    metric for metric, r in guardrails.items()
    if r['absolute_change_pp'] > 0 and r['p_value'] < ALPHA
]

print('Experiment health OK:', srm_ok)
print('Primary passes statistical + business thresholds:', primary_ok)
print('Significant adverse guardrails:', adverse_guardrails or 'None')

if not srm_ok:
    recommendation = 'DO NOT DECIDE — investigate experiment assignment first.'
elif primary_ok and not adverse_guardrails:
    recommendation = 'ROLL OUT / CONTROLLED ROLLOUT — benefit is credible, meaningful, and guardrails are acceptable.'
elif primary.p_value < ALPHA:
    recommendation = 'ITERATE / REASSESS — statistically credible result, but business threshold or guardrails need review.'
else:
    recommendation = 'NO FULL ROLLOUT YET — evidence is insufficient; review power, confidence interval, and treatment design.'

print('\nRecommendation:', recommendation)

## 14. Executive readout structure

### What happened
State the primary result in business terms.

### How confident are we?
Give the confidence interval and statistical conclusion without unnecessary jargon.

### What else changed?
Summarize secondary and guardrail metrics.

### What does it mean?
Translate the effect to business scale.

### Recommendation
Roll out, targeted rollout, iterate/retest, extend, or stop.

### Monitoring
Define the post-decision KPIs and time horizon.

## 15. Analyst QA checklist

- [ ] Business question and hypothesis defined before analysis.
- [ ] Primary KPI and guardrails pre-defined.
- [ ] Population and exclusions documented.
- [ ] Assignment and SRM validated.
- [ ] Data-quality issues quantified before cleaning.
- [ ] Statistical method matches metric type.
- [ ] Confidence intervals and effect sizes reported.
- [ ] Statistical and business significance separated.
- [ ] Segment findings labeled pre-specified vs exploratory.
- [ ] Guardrails and trade-offs discussed.
- [ ] Recommendation tied directly to evidence.
- [ ] Post-rollout monitoring defined.

## 16. Practice questions

1. Would you trust this experiment? Why?
2. What is the primary effect in absolute and relative terms?
3. Does the confidence interval support the business threshold?
4. Which guardrail deserves the most cautious interpretation?
5. Why is a two-proportion test appropriate for checkout completion?
6. Why inspect both mean and median checkout time?
7. Is the Mobile result confirmatory or exploratory?
8. What would make you choose a targeted rollout?
9. What additional financial inputs would you request for ROI?
10. Explain the recommendation to a non-technical client in three sentences.